In [38]:
import lzma
import pickle
import re
import unicodedata

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

In [39]:
with lzma.open("../../data/cleaned/starling_cleaned.pkl.xz", "rb") as f:
    starling = pickle.load(f)

with lzma.open("../../data/cleaned/unihan_cleaned.pkl.xz", "rb") as f:
    unihan = pickle.load(f)

In [40]:
import opencc

# Valid pinyin: a-z plus toned a/e/i/o/u/ü
_VALID_PINYIN_CHARS = re.compile(
    r'[^a-zāáǎàēéěèīíǐìōóǒòūúǔùüǖǘǚǜ]'
)

def take_non_missing_value(row):
    return "; ".join({unicodedata.normalize("NFC", str(value)) for value in row if pd.notna(value)})

def split_pinyin(s):
    parts = re.split(r'[;\s]+', s.strip())
    seen = set()
    result = []
    for p in parts:
        p = p.strip()
        if p and p not in seen:
            seen.add(p)
            result.append(p)
    return result if result else [s]

def clean_pinyin(s):
    """Remove invalid characters; return None if nothing valid remains."""
    cleaned = _VALID_PINYIN_CHARS.sub('', s)
    return cleaned if cleaned else None

_converter = opencc.OpenCC("t2s")

# Build radical map from kRSUnicode in Unihan_IRGSources.txt
# Format: radical_number.strokes  (prime suffix = simplified-form radical, still same number)
_radical_map: dict[str, str] = {}
with open("../../data/raw/Unihan_IRGSources.txt") as _f:
    for _line in _f:
        if _line.startswith('#') or not _line.strip():
            continue
        _parts = _line.strip().split('\t')
        if len(_parts) == 3 and _parts[1] == 'kRSUnicode':
            _m = re.match(r"(\d+)'?\.", _parts[2].split()[0])
            if _m:
                _char = chr(int(_parts[0][2:], 16))
                _radical_char = chr(0x2F00 + int(_m.group(1)) - 1)
                _radical_map[_char] = _radical_char

# Pinyin decomposition into (initial, final, tone)
_TONE_MARKS = {
    '\u0304': 1,  # macron  ā ē ī ō ū ǖ → tone 1
    '\u0301': 2,  # acute   á é í ó ú ǘ → tone 2
    '\u030c': 3,  # caron   ǎ ě ǐ ǒ ǔ ǚ → tone 3
    '\u0300': 4,  # grave   à è ì ò ù ǜ → tone 4
}
# Longest matches first so zh/ch/sh beat z/c/s
_INITIALS = [
    'zh', 'ch', 'sh',
    'b', 'p', 'm', 'f',
    'd', 't', 'n', 'l',
    'g', 'k', 'h',
    'j', 'q', 'x',
    'r', 'z', 'c', 's',
    'y', 'w',
]

def decompose_pinyin(s: str) -> str | None:
    """Return str(tuple) of (initial, final, tone) for a single pinyin syllable.
    Tone 5 means neutral/toneless."""
    if not s:
        return None
    nfd = unicodedata.normalize("NFD", s)
    # Extract tone number from the first combining tone mark found
    tone = 5
    for ch in nfd:
        t = _TONE_MARKS.get(ch)
        if t:
            tone = t
            break
    # Strip all combining characters to get the bare toneless syllable
    toneless = "".join(c for c in nfd if unicodedata.category(c) != "Mn")
    # Split off initial consonant(s)
    initial = ""
    for init in _INITIALS:
        if toneless.startswith(init):
            initial = init
            break
    final = toneless[len(initial):]
    return str((initial, final, tone))

merged = unihan.merge(starling, on="Character", how="outer", suffixes=("_unihan", "_starling"))

merged["definition"] = merged[["kDefinition", "English meaning"]].apply(take_non_missing_value, axis=1)
merged["pinyin"]     = merged[["kMandarin", "Modern (Beijing) reading"]].apply(take_non_missing_value, axis=1)

merged = merged.rename(columns={
    "Character":   "hanzi",
    "kHangul":     "hangul",
    "kKorean":     "anglo_hangul",
    "kJapanese":   "katakana",
    "kJapaneseOn": "anglo_katakana",
})[["hanzi", "definition", "pinyin", "hangul", "anglo_hangul", "katakana", "anglo_katakana"]]

merged["simplified"] = merged["hanzi"].apply(lambda x: _converter.convert(x) if pd.notna(x) else x)
merged["radical"]    = merged["hanzi"].map(_radical_map)

# Split pinyin on ";" and whitespace, deduplicate, explode to one reading per row
merged["pinyin"] = merged["pinyin"].apply(split_pinyin)
merged = merged.explode("pinyin").reset_index(drop=True)

# Strip invalid characters, drop rows with nothing left
merged["pinyin"] = merged["pinyin"].apply(clean_pinyin)
merged = merged.dropna(subset=["pinyin"]).reset_index(drop=True)

# Decompose pinyin into (initial, final, tone) tuple string
merged["decomposed_pinyin"] = merged["pinyin"].apply(decompose_pinyin)

# Toneless pinyin: strip combining tone marks from NFD form
merged["toneless_pinyin"] = merged["pinyin"].apply(
    lambda x: "".join(c for c in unicodedata.normalize("NFD", x) if unicodedata.category(c) != "Mn")
)

merged = merged[["hanzi", "simplified", "radical", "definition", "pinyin", "decomposed_pinyin", "toneless_pinyin", "hangul", "anglo_hangul", "katakana", "anglo_katakana"]]

merged


,hanzi,simplified,radical,definition,pinyin,decomposed_pinyin,toneless_pinyin,hangul,anglo_hangul,katakana,anglo_katakana
0,Ф,Ф,NaN,"to send, cause (?)",bēng,"('b', 'eng', 1)",beng,NaN,NaN,NaN,NaN
1,Щ,Щ,NaN,"be robust, strong",bì,"('b', 'i', 4)",bi,NaN,NaN,NaN,NaN
2,к,к,NaN,"flask, bottle (for wine)",yǒu,"('y', 'ou', 3)",you,NaN,NaN,NaN,NaN
3,へ,へ,NaN,"be grieved, sad",daō,"('d', 'ao', 1)",dao,NaN,NaN,NaN,NaN
4,一,一,⼀,"be one, single, whole; one; a, an; alone",yī,"('y', 'i', 1)",yi,일,IL,イチ,ICHI
...,...,...,...,...,...,...,...,...,...,...,...
8560,龜,龟,⿔,"turtle, tortoise; bone oracle in general; turt...",guī,"('g', 'ui', 1)",gui,구,KWU,キ,KI
8561,龝,龝,⿔,"autumn, fall; year",qiū,"('q', 'iu', 1)",qiu,추,CHWU,シュウ,SHUU
8562,龠,龠,⿕,"flute; pipe, ancient measure; Kangxi radical 214",yuè,"('y', 'ue', 4)",yue,약,YAK,ヤク,YAKU
8563,龢,龢,⿕,"in harmony; calm, peaceful",hé,"('h', 'e', 2)",he,화,HWA,カ,KA


In [41]:
import json
from pathlib import Path

import ollama

ANNO_MODEL = "qwen2.5:7b"
ANNO_CACHE = Path("../../data/cleaned/annotations.json")
ANNO_PROMPT = """\
You are an expert in Chinese language and culture.
Given a Chinese character, reply with a single JSON object — no markdown, no extra text — with exactly these keys:
  "meaning":   core English meaning (<=8 words)
  "name_use":  suitability as a given-name character: "good", "neutral", or "avoid"
  "note":      important cultural/negative connotation, or "" if none (<=10 words)

Character: {char}
Pinyin: {pinyin}
Dictionary definition: {definition}
"""

# Load existing cache so we can resume interrupted runs
_anno_cache: dict[str, dict] = {}
if ANNO_CACHE.exists():
    _anno_cache = json.loads(ANNO_CACHE.read_text())

def _annotate(char: str, pinyin: str, definition: str) -> dict:
    key = f"{char}|{pinyin}"
    if key in _anno_cache:
        return _anno_cache[key]
    prompt = ANNO_PROMPT.format(char=char, pinyin=pinyin, definition=definition)
    resp = ollama.chat(
        model=ANNO_MODEL,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0},
    )
    text = resp.message.content.strip()
    try:
        result = json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r'\{.*\}', text, re.DOTALL)
        result = json.loads(m.group()) if m else {"meaning": "", "name_use": "", "note": text[:80]}
    _anno_cache[key] = result
    return result

# Annotate unique (hanzi, pinyin) pairs — safe to re-run, resumes from cache
unique_chars = merged[["hanzi", "pinyin", "definition"]].drop_duplicates(subset=["hanzi", "pinyin"])
total = len(unique_chars)
for i, (_, row) in enumerate(unique_chars.iterrows()):
    key = f"{row['hanzi']}|{row['pinyin']}"
    if key not in _anno_cache:
        _annotate(row["hanzi"], row["pinyin"], row["definition"])
        ANNO_CACHE.write_text(json.dumps(_anno_cache, ensure_ascii=False, indent=2))
    if (i + 1) % 100 == 0 or (i + 1) == total:
        print(f"  {i+1}/{total} annotated")

print("Done.")


  100/8562 annotated
  200/8562 annotated
  300/8562 annotated
  400/8562 annotated
  500/8562 annotated
  600/8562 annotated
  700/8562 annotated
  800/8562 annotated
  900/8562 annotated
  1000/8562 annotated
  1100/8562 annotated
  1200/8562 annotated
  1300/8562 annotated
  1400/8562 annotated
  1500/8562 annotated
  1600/8562 annotated
  1700/8562 annotated
  1800/8562 annotated
  1900/8562 annotated
  2000/8562 annotated
  2100/8562 annotated
  2200/8562 annotated
  2300/8562 annotated
  2400/8562 annotated
  2500/8562 annotated
  2600/8562 annotated
  2700/8562 annotated
  2800/8562 annotated
  2900/8562 annotated
  3000/8562 annotated
  3100/8562 annotated
  3200/8562 annotated
  3300/8562 annotated
  3400/8562 annotated
  3500/8562 annotated
  3600/8562 annotated
  3700/8562 annotated
  3800/8562 annotated
  3900/8562 annotated
  4000/8562 annotated
  4100/8562 annotated
  4200/8562 annotated
  4300/8562 annotated
  4400/8562 annotated
  4500/8562 annotated
  4600/8562 annotat

In [42]:
def _get_anno(hanzi, pinyin, field):
    return _anno_cache.get(f"{hanzi}|{pinyin}", {}).get(field)

merged["meaning"]  = merged.apply(lambda r: _get_anno(r["hanzi"], r["pinyin"], "meaning"),  axis=1)
merged["name_use"] = merged.apply(lambda r: _get_anno(r["hanzi"], r["pinyin"], "name_use"), axis=1)
merged["note"]     = merged.apply(lambda r: _get_anno(r["hanzi"], r["pinyin"], "note"),     axis=1)

merged


,hanzi,simplified,radical,definition,pinyin,decomposed_pinyin,toneless_pinyin,hangul,anglo_hangul,katakana,anglo_katakana,meaning,name_use,note
0,Ф,Ф,NaN,"to send, cause (?)",bēng,"('b', 'eng', 1)",beng,NaN,NaN,NaN,NaN,to collapse,avoid,negative connotation
1,Щ,Щ,NaN,"be robust, strong",bì,"('b', 'i', 4)",bi,NaN,NaN,NaN,NaN,"robust, strong",avoid,Not a Chinese character
2,к,к,NaN,"flask, bottle (for wine)",yǒu,"('y', 'ou', 3)",you,NaN,NaN,NaN,NaN,bottle,avoid,不吉利
3,へ,へ,NaN,"be grieved, sad",daō,"('d', 'ao', 1)",dao,NaN,NaN,NaN,NaN,"sad, grieve",avoid,negative connotation
4,一,一,⼀,"be one, single, whole; one; a, an; alone",yī,"('y', 'i', 1)",yi,일,IL,イチ,ICHI,one; single,neutral,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8560,龜,龟,⿔,"turtle, tortoise; bone oracle in general; turt...",guī,"('g', 'ui', 1)",gui,구,KWU,キ,KI,"turtle, tortoise",neutral,
8561,龝,龝,⿔,"autumn, fall; year",qiū,"('q', 'iu', 1)",qiu,추,CHWU,シュウ,SHUU,autumn; year,avoid,negative connotation
8562,龠,龠,⿕,"flute; pipe, ancient measure; Kangxi radical 214",yuè,"('y', 'ue', 4)",yue,약,YAK,ヤク,YAKU,ancient measure; flute,avoid,negative connotation
8563,龢,龢,⿕,"in harmony; calm, peaceful",hé,"('h', 'e', 2)",he,화,HWA,カ,KA,in harmony,neutral,


In [43]:
merged.to_pickle("../../data/cleaned/merged.pkl.xz")